# Детальный разбор кода предобработки данных

## Блок 1: Импорт библиотек

```python
import pandas as pd
import numpy as np
import re
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor
import joblib
import warnings
warnings.filterwarnings('ignore')
```

**Что делает:**
- `pandas`, `numpy` - основные библиотеки для работы с данными
- `re` - регулярные выражения для очистки текста
- `sklearn.cluster` - алгоритмы кластеризации (K-means)
- `sklearn.preprocessing` - стандартизация данных
- `sklearn.feature_extraction.text` - преобразование текста в числовые признаки
- `sklearn.decomposition` - уменьшение размерности (SVD)
- `nltk.sentiment` - анализ тональности текста
- `catboost` - алгоритм машинного обучения (хотя в коде не используется)
- `joblib` - сохранение и загрузка моделей
- `warnings` - подавление предупреждений

## Блок 2: Загрузка и начальная обработка данных

```python
df = pd.read_csv('/kaggle/input/vreros-dataset-a/train.tsv', sep='\t')
df = df[df['target'] > 0].reset_index(drop=True)
```

**Что делает:**
- Загружает данные из TSV-файла с разделителем табуляции
- Фильтрует строки, оставляя только те, где целевая переменная `target` больше 0
- Сбрасывает индекс после фильтрации

## Блок 3: Географические признаки

### Подблок 3.1: Парсинг координат

```python
df['coordinates'] = df['coordinates'].apply(lambda x: eval(x) if isinstance(x, str) else x)
df['longitude'] = df['coordinates'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 2 else np.nan)
df['latitude'] = df['coordinates'].apply(lambda x: x[1] if isinstance(x, list) and len(x) == 2 else np.nan)
```

**Что делает:**
- Преобразует строковое представление координат в Python-объекты с помощью `eval()`
- Извлекает долготу (первый элемент списка) и широту (второй элемент)
- Обрабатывает случаи с некорректными данными, устанавливая `np.nan`

### Подблок 3.2: Расчет расстояния до центра

```python
MOSCOW_CENTER = (55.7558, 37.6176)
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Радиус Земли в км
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

df['distance_to_center'] = df.apply(
    lambda row: haversine_distance(row['latitude'], row['longitude'], 
                                 MOSCOW_CENTER[0], MOSCOW_CENTER[1])
    if pd.notna(row['latitude']) and pd.notna(row['longitude']) else 20.0, 
    axis=1
)
```

**Что делает:**
- Реализует формулу Хаверсина для расчета расстояния между двумя точками на сфере
- Для каждой строки вычисляет расстояние до центра Москвы
- Если координаты отсутствуют, устанавливает расстояние 20.0 км по умолчанию

### Подблок 3.3: Производные географические признаки

```python
df['lat_sq'] = df['latitude'] ** 2
df['lon_sq'] = df['longitude'] ** 2
df['lat_lon_product'] = df['latitude'] * df['longitude']
```

**Что делает:**
- Создает квадраты координат для учета нелинейных зависимостей
- Создает произведение координат для учета взаимодействия между широтой и долготой

## Блок 4: Кластеризация

### Подблок 4.1: Географическая кластеризация

```python
coords = df[['latitude', 'longitude']].dropna()
geo_kmeans = KMeans(n_clusters=12, random_state=42, n_init=10, init='k-means++').fit(coords)
df['geo_cluster'] = np.nan
df.loc[coords.index, 'geo_cluster'] = geo_kmeans.predict(coords)
df['geo_cluster'] = df['geo_cluster'].fillna(-1).astype(int)
```

**Что делает:**
- Выбирает координаты и удаляет пропущенные значения
- Обучает K-means модель с 12 кластерами на географических данных
- Присваивает каждому объекту кластер, заполняет пропуски значением -1

### Подблок 4.2: Кластеризация по признакам

```python
all_numeric_features = [col for col in df.columns if 
                       col not in ['id', 'name', 'address', 'coordinates', 'target'] and 
                       df[col].dtype in ['float64', 'int64']]

radius_1000m_features = [f for f in all_numeric_features if '_1000m' in f]
additional_features = ['distance_to_center', 'latitude', 'longitude', 'lat_sq', 'lon_sq', 'lat_lon_product']

features_for_clustering = radius_1000m_features + additional_features

scaler = StandardScaler()
X_cluster = scaler.fit_transform(df[features_for_clustering].fillna(0))
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10, init='k-means++').fit(X_cluster)
df['feature_cluster'] = kmeans.predict(X_cluster)
```

**Что делает:**
- Собирает все числовые признаки, исключая служебные колонки
- Отбирает признаки с суффиксом '_1000m' (предположительно, признаки в радиусе 1000 метров)
- Добавляет дополнительные географические признаки
- Стандартизирует данные (приводит к нулевому среднему и единичной дисперсии)
- Обучает K-means модель с 10 кластерами на признаках

### Подблок 4.3: Сохранение моделей кластеризации

```python
joblib.dump(geo_kmeans, 'geo_kmeans.pkl')
joblib.dump(kmeans, 'feature_kmeans.pkl')
joblib.dump(scaler, 'cluster_scaler.pkl')
```

**Что делает:**
- Сохраняет обученные модели и scaler для использования на новых данных

## Блок 5: Текстовые признаки

### Подблок 5.1: Загрузка и базовая обработка отзывов

```python
reviews = pd.read_csv('/kaggle/input/vreros-dataset-a/reviews.txv/reviews.tsv', sep='\t')

review_stats = reviews.groupby('id').agg(
    review_count=('text', 'count'),
    avg_review_length=('text', lambda x: x.str.len().mean())
).reset_index()
```

**Что делает:**
- Загружает данные отзывов
- Агрегирует отзывы по ID заведения:
  - `review_count` - количество отзывов
  - `avg_review_length` - средняя длина отзыва

### Подблок 5.2: Очистка текста и TF-IDF

```python
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Zа-яА-ЯёЁ0-9\s]', '', text)
    return text

reviews['clean_text'] = reviews['text'].apply(clean_text)
reviews['text_length'] = reviews['text'].str.len()

text_grouped = reviews.groupby('id')['clean_text'].apply(lambda x: ' '.join(x)).reset_index()

tfidf = TfidfVectorizer(max_features=500, min_df=0.01, max_df=0.95)
tfidf_matrix = tfidf.fit_transform(text_grouped['clean_text'])
```

**Что делает:**
- Функция `clean_text` приводит текст к нижнему регистру и удаляет все символы, кроме букв и цифр
- Объединяет все отзывы по каждому заведению в один текст
- Создает TF-IDF матрицу с ограничением в 500 признаков
- `min_df=0.01` - игнорирует слова, встречающиеся менее чем в 1% документов
- `max_df=0.95` - игнорирует слова, встречающиеся более чем в 95% документов

### Подблок 5.3: Уменьшение размерности текстовых признаков

```python
svd = TruncatedSVD(n_components=30, random_state=42)
text_features = svd.fit_transform(tfidf_matrix)

text_feature_names = [f'text_feature_{i}' for i in range(text_features.shape[1])]
text_features_df = pd.DataFrame(text_features, columns=text_feature_names)
text_features_df['id'] = text_grouped['id']
```

**Что делает:**
- Уменьшает размерность TF-IDF матрицы с 500 до 30 компонент с помощью SVD
- Создает DataFrame с текстовыми признаками

### Подблок 5.4: Анализ тональности

```python
sia = SentimentIntensityAnalyzer()
def get_sentiment(text):
    if not isinstance(text, str) or len(text) < 10:
        return 0.0
    return sia.polarity_scores(text)['compound']

sentiment = reviews.groupby('id')['text'].apply(
    lambda x: x.apply(get_sentiment).mean()
).reset_index(name='avg_sentiment')
```

**Что делает:**
- Использует VADER для анализа эмоциональной окраски текста
- Вычисляет compound score (обобщенный показатель тональности от -1 до 1)
- Для коротких или некорректных текстов возвращает 0
- Вычисляет среднюю тональность по всем отзывам заведения

### Подблок 5.5: Объединение текстовых признаков

```python
text_features = pd.merge(text_features_df, review_stats, on='id', how='left')
text_features = pd.merge(text_features, sentiment, on='id', how='left')
text_features = text_features.fillna(0)
```

**Что делает:**
- Объединяет все текстовые признаки в один DataFrame
- Заполняет пропуски нулями для заведений без отзывов

## Блок 6: Агрегация статистик по кластерам

```python
geo_cluster_stats = df.groupby('geo_cluster').agg({
    'target': ['mean', 'std', 'count', 'median']
}).round(4)
geo_cluster_stats.columns = ['geo_mean', 'geo_std', 'geo_count', 'geo_median']

feature_cluster_stats = df.groupby('feature_cluster').agg({
    'target': ['mean', 'std', 'count', 'median']
}).round(4)
feature_cluster_stats.columns = ['feature_mean', 'feature_std', 'feature_count', 'feature_median']
```

**Что делает:**
- Для каждого географического кластера вычисляет статистики по целевой переменной:
  - Среднее значение рейтинга
  - Стандартное отклонение
  - Количество объектов
  - Медиану
- Аналогично для кластеров по признакам

## Блок 7: Функция отбора признаков по корреляции

```python
def select_features_by_correlation(df, target_col, correlation_threshold=0.9):
    # Выбираем только числовые признаки
    numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_features = [f for f in numeric_features if f != target_col]
    
    # ИСКЛЮЧАЕМ ПРИЗНАКИ С ID
    excluded_features = ['id', 'id.1']
    special_features = ['review_count', 'avg_review_length']
    
    numeric_features = [f for f in numeric_features if f not in excluded_features]
    
    # Вычисляем корреляционную матрицу
    corr_matrix = df[numeric_features + [target_col]].corr().abs()
    
    # Верхний треугольник матрицы корреляции
    upper_triangle = corr_matrix.where(np.triu(np.ones_like(corr_matrix, dtype=bool), k=1))
    
    # Находим пары признаков с корреляцией выше порога
    high_corr_pairs = []
    for col in upper_triangle.columns:
        high_corr = upper_triangle[col][upper_triangle[col] > correlation_threshold]
        for row in high_corr.index:
            high_corr_pairs.append((col, row, high_corr[row]))
    
    # Группируем сильно коррелирующие признаки
    feature_groups = {}
    for feat1, feat2, corr_value in high_corr_pairs:
        group_found = False
        for group in feature_groups.values():
            if feat1 in group or feat2 in group:
                group.update([feat1, feat2])
                group_found = True
                break
        if not group_found:
            feature_groups[len(feature_groups)] = {feat1, feat2}
    
    # Для каждой группы оставляем признак с наибольшей корреляцией с целевой переменной
    features_to_keep = set()
    features_to_remove = set()
    
    for group in feature_groups.values():
        best_feature = None
        best_corr = -1
        
        for feature in group:
            if feature in special_features:
                continue
            corr_with_target = corr_matrix.loc[feature, target_col]
            if corr_with_target > best_corr:
                best_corr = corr_with_target
                best_feature = feature
        
        if best_feature:
            features_to_keep.add(best_feature)
            features_to_remove.update([f for f in group if f != best_feature])
    
    # Все признаки, которые не попали в группы сильно коррелирующих, тоже оставляем
    all_features_set = set(numeric_features)
    independent_features = all_features_set - set().union(*feature_groups.values())
    features_to_keep.update(independent_features)
    
    # Удаляем признаки, которые должны быть удалены, но сохраняем особые признаки
    features_to_remove = features_to_remove - set(special_features)
    final_features = list(features_to_keep - features_to_remove)
    
    # ДОБАВЛЯЕМ ОСОБЫЕ ПРИЗНАКИ
    for special_feat in special_features:
        if special_feat in df.columns and special_feat not in final_features:
            final_features.append(special_feat)
    
    return final_features
```

**Что делает:**
1. **Отбор числовых признаков**: Исключает нечисловые колонки и целевую переменную
2. **Исключение служебных колонок**: Удаляет ID-колонки
3. **Вычисление корреляций**: Строит матрицу абсолютных корреляций
4. **Поиск сильно коррелирующих пар**: Находит пары признаков с корреляцией > 0.9
5. **Группировка коррелирующих признаков**: Объединяет сильно коррелирующие признаки в группы
6. **Отбор лучшего признака из группы**: В каждой группе оставляет признак с наибольшей корреляцией с целевой переменной
7. **Сохранение особых признаков**: Гарантирует сохранение `review_count` и `avg_review_length`

## Блок 8: Финальная обработка и сохранение

```python
# Выполнение отбора признаков
selected_features = select_features_by_correlation(df, 'target', correlation_threshold=0.9)

# Добавление категориальных признаков
categorical_features = ['category', 'geo_cluster', 'feature_cluster', 'name', 'address']
selected_features.extend([f for f in categorical_features if f in df.columns])

# Удаление дубликатов и ID-колонок
selected_features = list(set(selected_features))
selected_features = [f for f in selected_features if 'id' not in f.lower()]

# Создание финального датафрейма
id_columns = [col for col in df.columns if 'id' in col.lower()]
features_without_id = [f for f in selected_features if f not in id_columns]
df_filtered = df[['target'] + features_without_id].copy()

# Очистка названий столбцов для совместимости с XGBoost
def clean_column_names(df):
    df_clean = df.copy()
    df_clean.columns = [col.replace('[', '_').replace(']', '_').replace('<', '_lt_').replace('>', '_gt_') 
                       for col in df_clean.columns]
    return df_clean

df_filtered = clean_column_names(df_filtered)
```

**Что делает:**
- Применяет функцию отбора признаков
- Добавляет категориальные признаки, которые не были учтены в числовом отборе
- Удаляет дубликаты и ID-колонки из финального набора
- Очищает названия столбцов от символов, которые могут вызвать проблемы в XGBoost

## Итоговый результат

Код создает комплексный набор признаков, который включает:
- **Географические признаки** (координаты, расстояние до центра)
- **Кластерные признаки** (географические и по признакам)
- **Текстовые признаки** (TF-IDF, статистики отзывов, тональность)
- **Статистики по кластерам** (средние рейтинги по кластерам)
- **Отобранные признаки** (удалены сильно коррелирующие)

Все модели и преобразования сохраняются для использования на тестовых данных.